## Explore full-catalog data (SearchItems, 10k listings)

Load the newly scraped full-catalog dataset and check its basic shape
and missing values, before cleaning.

In [1]:
import pandas as pd

df = pd.read_csv("../data/items_full.csv")
df.shape

(10008, 12)

Check basic stats and missing values before cleaning.

In [2]:
print(df.isna().sum())
df.describe()

id              0
price           0
currency        0
rooms           0
area            0
floor           0
floors          0
hasRepair      14
isVipped        0
isFeatured      0
location      465
city            0
dtype: int64


,id,price,rooms,area,floor,floors
count,1.000800e+04,1.000800e+04,10008.000000,10008.000000,10008.000000,10008.000000
mean,6.328983e+06,2.879092e+05,2.715128,97.236093,7.358413,12.819844
std,3.212801e+05,2.033362e+05,0.898064,56.661427,4.701751,5.380145
min,3.049175e+06,1.900000e+04,1.000000,15.000000,1.000000,1.000000
25%,6.315260e+06,1.660000e+05,2.000000,62.500000,4.000000,9.000000
50%,6.447973e+06,2.350000e+05,3.000000,85.000000,6.000000,14.000000
75%,6.489092e+06,3.450000e+05,3.000000,119.000000,10.000000,17.000000
max,6.494899e+06,4.020000e+06,19.000000,1000.000000,28.000000,33.000000


Clean the data using the same logic as scripts/clean_data.py: drop
missing rooms/location, filter price outliers via price_per_m2.

In [3]:
df_clean = df[df["rooms"].notna()].copy()
df_clean = df_clean[df_clean["price"] >= 5000]
df_clean["price_per_m2"] = df_clean["price"] / df_clean["area"]
df_clean = df_clean[df_clean["price_per_m2"] >= 300]
df_clean["rooms"] = df_clean["rooms"].astype(int)
df_clean = df_clean[df_clean["location"].notna()]
df_clean["price_per_m2"] = df_clean["price_per_m2"].round(2)

df_clean.shape

(9541, 13)

Test the hypothesis: does isVipped correlate with price, controlling
for rooms/area (since VIP listings might just be bigger/pricier
apartments in general, not cheaper/pricier for the same size).

In [4]:
df_clean.groupby("isVipped")[["price", "price_per_m2", "area", "rooms"]].mean()

,price,price_per_m2,area,rooms
isVipped,,,,
False,288352.410126,2979.894026,96.965256,2.730314
True,329815.635119,3140.631589,104.322333,2.731548


Save cleaned full-catalog data for use in model training.

In [5]:
df_clean.to_csv("../data/item_clean_full.csv", index=False, encoding="utf-8")

Prepare features: group rare locations, one-hot encode, split train/test.

In [6]:
location_counts = df_clean["location"].value_counts()
common_locations = location_counts[location_counts >= 5].index
df_clean["location"] = df_clean["location"].where(df_clean["location"].isin(common_locations), "Other")

df_clean["location"].nunique()

60

Check how much "Other" shrank compared to the old dataset.

In [7]:
df_clean["location"].value_counts()["Other"]

np.int64(29)

One-hot encode location.

In [8]:
df_encoded = pd.get_dummies(df_clean, columns=["location"])
df_encoded.shape

(9541, 72)

Split into features and target. Exclude id, price, currency,
price_per_m2 (target-derived), city (little variance). Keep isVipped
and isFeatured as features (correlate with price, likely via
self-selection — see analysis above); the API will fix these to a
constant value rather than exposing them in the form.

In [9]:
features = df_encoded.drop(columns=["id", "price", "currency", "price_per_m2", "city"])
target = df_encoded["price"]

features.shape, target.shape

((9541, 67), (9541,))

Check and fill remaining missing values before training.

In [10]:
print(features.isna().sum()[features.isna().sum() > 0])

features["hasRepair"] = features["hasRepair"].fillna(features["hasRepair"].mode()[0])

features.isna().sum().sum()

hasRepair    14
dtype: int64


np.int64(0)

Split into train/test sets (80/20), same approach as before.

In [11]:
from sklearn.model_selection import train_test_split

features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size=0.2, random_state=42
)

features_train.shape, features_test.shape

((7632, 67), (1909, 67))

Define evaluation function and train/compare the same 4 candidate models.

In [12]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

def evaluate(model, features_test, target_test):
    predictions = model.predict(features_test)
    return {
        "MAE": mean_absolute_error(target_test, predictions),
        "RMSE": root_mean_squared_error(target_test, predictions),
        "R2": r2_score(target_test, predictions),
    }

candidates = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

results = {}
for name, model in candidates.items():
    model.fit(features_train, target_train)
    results[name] = evaluate(model, features_test, target_test)

results_df = pd.DataFrame(results).T
results_df

,MAE,RMSE,R2
Linear Regression,56559.452445,116521.727642,0.686552
Ridge,56485.037637,116572.135516,0.686280
Random Forest,42038.776613,96573.860597,0.784686
Gradient Boosting,54248.111535,102212.220618,0.758811


In [13]:
target_test.mean()

np.float64(298901.1215295966)

Tune Random Forest hyperparameters via grid search with cross-validation
on the larger dataset.

In [14]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    scoring="neg_mean_absolute_error",
    cv=5,
)
grid_search.fit(features_train, target_train)

grid_search.best_params_

{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 300}

Evaluate the tuned model on the test set.

In [15]:
best_rf_model = grid_search.best_estimator_

tuned_results = evaluate(best_rf_model, features_test, target_test)
tuned_results

{'MAE': 41726.0917781484, 'RMSE': 96339.19768093561, 'R2': 0.7857314724787801}

Evaluate the tuned model on the test set.

In [16]:
import joblib

joblib.dump(best_rf_model, "../models/model.pkl")

['../models/model.pkl']

In [17]:
list(best_rf_model.feature_names_in_)

['rooms',
 'area',
 'floor',
 'floors',
 'hasRepair',
 'isVipped',
 'isFeatured',
 'location_20 Yanvar',
 'location_28 May',
 'location_7-ci mikrorayon',
 'location_8 Noyabr',
 'location_8-ci kilometr',
 'location_8-ci mikrorayon',
 'location_9-cu mikrorayon',
 'location_Abşeron',
 'location_Avtovağzal',
 'location_Azadlıq Prospekti',
 'location_Ağ şəhər',
 'location_Badamdar',
 'location_Bakmil',
 'location_Bakıxanov',
 'location_Bayıl',
 'location_Biləcəri',
 'location_Binə',
 'location_Binəqədi',
 'location_Buzovna',
 'location_Dərnəgül',
 'location_Elmlər Akademiyası',
 'location_Gənclik',
 'location_Hövsan',
 'location_Həzi Aslanov',
 'location_Koroğlu',
 'location_Köhnə Günəşli',
 'location_Lökbatan',
 'location_M.Ə.Rəsulzadə',
 'location_Masazır',
 'location_Massiv D',
 'location_Memar Əcəmi',
 'location_Məmmədli',
 'location_Mərdəkan',
 'location_Nardaran',
 'location_Neftçilər',
 'location_Nizami',
 'location_Nəriman Nərimanov',
 'location_Nərimanov',
 'location_Nəsimi',
 'loc

Compare model size vs MAE across different n_estimators, to find
a practical tradeoff (GitHub has a 100MB file size limit).

In [18]:
import joblib
import os

for n in [50, 100, 150, 200, 300]:
    model = RandomForestRegressor(n_estimators=n, random_state=42)
    model.fit(features_train, target_train)

    joblib.dump(model, "../models/test_model.pkl")
    size_mb = os.path.getsize("../models/test_model.pkl") / (1024 * 1024)

    mae = evaluate(model, features_test, target_test)["MAE"]

    print(f"n_estimators={n}: MAE={mae:.0f}, size={size_mb:.1f} MB")

os.remove("../models/test_model.pkl")

n_estimators=50: MAE=42047, size=27.9 MB
n_estimators=100: MAE=42039, size=55.7 MB
n_estimators=150: MAE=41958, size=83.6 MB
n_estimators=200: MAE=41803, size=111.4 MB
n_estimators=300: MAE=41726, size=167.2 MB


Final model: n_estimators=50. Negligible MAE difference vs tuned
version (42047 vs 41726, ~0.76%) but 6x smaller file (28MB vs 167MB),
staying well under GitHub's 100MB limit.

In [19]:
final_model = RandomForestRegressor(n_estimators=50, random_state=42)
final_model.fit(features_train, target_train)

joblib.dump(final_model, "../models/model.pkl")

['../models/model.pkl']